# Linear Attention (线性注意力)

## 概述

线性注意力通过kernel trick将注意力的计算复杂度从O(n²)降低到O(n)，使得处理超长序列成为可能。

### 核心思想

**标准注意力:**
```
Attention(Q, K, V) = softmax(QK^T / √d) V
                   = D^(-1) exp(QK^T) V
```

**线性注意力:**
```
Attention(Q, K, V) = φ(Q) (φ(K)^T V) / (φ(Q) 1^T φ(K))
```

关键在于改变计算顺序：先计算 `φ(K)^T V`（形状d×d），避免显式构造n×n的注意力矩阵。

### 复杂度对比

| 方法 | 时间复杂度 | 空间复杂度 |
|------|-----------|----------|
| 标准注意力 | O(n²d) | O(n²) |
| 线性注意力 | O(nd²) ≈ O(n) | O(d²) |

当序列长度 n >> 嵌入维度 d 时，线性注意力有显著优势。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

## 1. 特征映射函数

线性注意力的关键是选择合适的特征映射函数 φ(x)，将Query和Key映射到高维空间。

In [ ]:
def elu_feature_map(x):
    """
    ELU+1特征映射
    简单但有效，保证输出非负
    """
    return np.maximum(0, x) + np.minimum(0, np.exp(x) - 1) + 1


def random_fourier_features(x, num_features=None, scale=1.0, seed=42):
    """
    随机傅里叶特征（Performer使用）
    使用随机投影近似RBF核
    """
    n, d = x.shape
    if num_features is None:
        num_features = 2 * d
    
    np.random.seed(seed)
    omega = np.random.randn(d, num_features // 2) * scale
    b = np.random.uniform(0, 2 * np.pi, num_features // 2)
    
    projection = np.dot(x, omega) + b
    features = np.concatenate([
        np.cos(projection),
        np.sin(projection)
    ], axis=-1)
    
    return features * np.sqrt(2.0 / num_features)


# 测试特征映射
test_input = np.random.randn(5, 8)

print("原始输入:")
print(test_input[0])
print("\nELU特征映射:")
print(elu_feature_map(test_input)[0])
print("\n随机傅里叶特征:")
print(random_fourier_features(test_input, num_features=16)[0])

## 2. 标准注意力实现（对比基准）

In [ ]:
def softmax(x, axis=-1):
    """Softmax函数"""
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


class StandardAttention:
    """
    标准注意力机制
    复杂度: O(n²d)
    内存: O(n²)
    """
    
    def __init__(self, embed_dim):
        self.embed_dim = embed_dim
        self.W_q = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_k = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_v = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
    
    def forward(self, x, return_attention=False):
        Q = np.dot(x, self.W_q)
        K = np.dot(x, self.W_k)
        V = np.dot(x, self.W_v)
        
        # 计算注意力矩阵 (n×n)
        scores = np.dot(Q, K.T) / np.sqrt(self.embed_dim)
        attention = softmax(scores, axis=-1)
        
        # 加权求和
        output = np.dot(attention, V)
        
        if return_attention:
            return output, attention
        return output

## 3. 线性注意力实现

In [ ]:
class LinearAttention:
    """
    线性注意力机制
    复杂度: O(nd²) ≈ O(n)
    内存: O(d²)
    """
    
    def __init__(self, embed_dim, feature_map='elu', num_features=None):
        self.embed_dim = embed_dim
        self.feature_map_type = feature_map
        self.num_features = num_features if num_features else embed_dim
        
        self.W_q = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_k = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_v = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
    
    def feature_map(self, x):
        """应用特征映射"""
        if self.feature_map_type == 'elu':
            return elu_feature_map(x)
        elif self.feature_map_type == 'rff':
            return random_fourier_features(x, self.num_features)
        else:
            raise ValueError(f"Unknown feature map: {self.feature_map_type}")
    
    def forward(self, x, return_kv=False):
        """
        线性注意力前向传播
        
        核心技巧：
        1. 先计算 K^T @ V (d×d矩阵)
        2. 再计算 Q @ (K^T @ V)
        避免构造 Q @ K^T (n×n矩阵)
        """
        Q = np.dot(x, self.W_q)
        K = np.dot(x, self.W_k)
        V = np.dot(x, self.W_v)
        
        # 应用特征映射
        Q_prime = self.feature_map(Q)
        K_prime = self.feature_map(K)
        
        # 关键步骤：先计算 K^T @ V (避免n×n矩阵)
        KV = np.dot(K_prime.T, V)  # (feature_dim, embed_dim)
        
        # 归一化分母
        K_sum = np.sum(K_prime, axis=0, keepdims=True)
        
        # 计算输出
        numerator = np.dot(Q_prime, KV)
        denominator = np.dot(Q_prime, K_sum.T)
        output = numerator / (denominator + 1e-6)
        
        if return_kv:
            return output, KV
        return output

## 4. 性能对比实验

In [ ]:
def benchmark_attention(seq_lengths, embed_dim=64):
    """
    比较标准注意力和线性注意力的性能
    """
    std_times = []
    linear_times = []
    
    for seq_len in seq_lengths:
        x = np.random.randn(seq_len, embed_dim)
        
        # 标准注意力
        std_attn = StandardAttention(embed_dim)
        start = time.time()
        _ = std_attn.forward(x)
        std_time = time.time() - start
        std_times.append(std_time)
        
        # 线性注意力
        linear_attn = LinearAttention(embed_dim, feature_map='elu')
        start = time.time()
        _ = linear_attn.forward(x)
        linear_time = time.time() - start
        linear_times.append(linear_time)
        
        print(f"序列长度 {seq_len:4d} - 标准: {std_time*1000:7.3f}ms, 线性: {linear_time*1000:7.3f}ms, 加速: {std_time/linear_time:.2f}x")
    
    return std_times, linear_times


# 运行基准测试
seq_lengths = [32, 64, 128, 256, 512]
std_times, linear_times = benchmark_attention(seq_lengths, embed_dim=32)

In [ ]:
# 绘制性能对比图
plt.figure(figsize=(12, 5))

# 时间对比
plt.subplot(1, 2, 1)
plt.plot(seq_lengths, [t*1000 for t in std_times], 'o-', label='标准注意力', linewidth=2)
plt.plot(seq_lengths, [t*1000 for t in linear_times], 's-', label='线性注意力', linewidth=2)
plt.xlabel('序列长度', fontsize=12)
plt.ylabel('时间 (ms)', fontsize=12)
plt.title('计算时间对比', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

# 加速比
plt.subplot(1, 2, 2)
speedups = [s/l for s, l in zip(std_times, linear_times)]
plt.plot(seq_lengths, speedups, 'd-', color='green', linewidth=2, markersize=8)
plt.axhline(y=1, color='r', linestyle='--', alpha=0.5, label='无加速')
plt.xlabel('序列长度', fontsize=12)
plt.ylabel('加速比', fontsize=12)
plt.title('线性注意力加速比', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 输出质量对比

线性注意力是标准注意力的近似，会有一定的精度损失。

In [ ]:
# 比较输出差异
seq_len = 64
embed_dim = 32
x = np.random.randn(seq_len, embed_dim)

# 标准注意力
std_attn = StandardAttention(embed_dim)
output_std = std_attn.forward(x)

# 线性注意力 (ELU)
linear_attn_elu = LinearAttention(embed_dim, feature_map='elu')
output_linear_elu = linear_attn_elu.forward(x)

# 线性注意力 (RFF)
linear_attn_rff = LinearAttention(embed_dim, feature_map='rff', num_features=64)
output_linear_rff = linear_attn_rff.forward(x)

# 计算差异
diff_elu = np.abs(output_std - output_linear_elu)
diff_rff = np.abs(output_std - output_linear_rff)

print("输出差异统计:")
print(f"\n线性注意力(ELU) vs 标准注意力:")
print(f"  平均绝对误差: {np.mean(diff_elu):.6f}")
print(f"  最大绝对误差: {np.max(diff_elu):.6f}")
print(f"  相对误差: {np.mean(diff_elu) / np.mean(np.abs(output_std)):.4%}")

print(f"\n线性注意力(RFF) vs 标准注意力:")
print(f"  平均绝对误差: {np.mean(diff_rff):.6f}")
print(f"  最大绝对误差: {np.max(diff_rff):.6f}")
print(f"  相对误差: {np.mean(diff_rff) / np.mean(np.abs(output_std)):.4%}")

In [ ]:
# 可视化输出差异
plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.imshow(output_std[:20, :20], cmap='RdBu_r', aspect='auto')
plt.colorbar()
plt.title('标准注意力输出', fontsize=12)
plt.xlabel('特征维度')
plt.ylabel('序列位置')

plt.subplot(1, 3, 2)
plt.imshow(output_linear_elu[:20, :20], cmap='RdBu_r', aspect='auto')
plt.colorbar()
plt.title('线性注意力(ELU)输出', fontsize=12)
plt.xlabel('特征维度')
plt.ylabel('序列位置')

plt.subplot(1, 3, 3)
plt.imshow(diff_elu[:20, :20], cmap='Reds', aspect='auto')
plt.colorbar()
plt.title('绝对误差', fontsize=12)
plt.xlabel('特征维度')
plt.ylabel('序列位置')

plt.tight_layout()
plt.show()

## 6. 内存占用对比

线性注意力最大的优势之一是显著降低内存占用。

In [ ]:
def estimate_memory(seq_len, embed_dim):
    """
    估算内存占用（仅考虑注意力矩阵）
    """
    # 标准注意力：需要存储 n×n 的注意力矩阵
    std_memory = seq_len * seq_len * 4  # float32 = 4 bytes
    
    # 线性注意力：只需要存储 d×d 的 KV 矩阵
    linear_memory = embed_dim * embed_dim * 4
    
    return std_memory, linear_memory


# 不同序列长度的内存占用
seq_lengths = [128, 256, 512, 1024, 2048, 4096]
embed_dim = 64

print(f"{'序列长度':<12} {'标准注意力(MB)':<18} {'线性注意力(MB)':<18} {'节省比例':<12}")
print("-" * 70)

std_mems = []
linear_mems = []

for seq_len in seq_lengths:
    std_mem, linear_mem = estimate_memory(seq_len, embed_dim)
    std_mb = std_mem / (1024 * 1024)
    linear_mb = linear_mem / (1024 * 1024)
    saving = (std_mem - linear_mem) / std_mem * 100
    
    std_mems.append(std_mb)
    linear_mems.append(linear_mb)
    
    print(f"{seq_len:<12} {std_mb:<18.4f} {linear_mb:<18.4f} {saving:<12.2f}%")

In [ ]:
# 绘制内存占用对比图
plt.figure(figsize=(10, 6))
plt.plot(seq_lengths, std_mems, 'o-', label='标准注意力', linewidth=2, markersize=8)
plt.plot(seq_lengths, linear_mems, 's-', label='线性注意力', linewidth=2, markersize=8)
plt.xlabel('序列长度', fontsize=12)
plt.ylabel('内存占用 (MB)', fontsize=12)
plt.title('注意力矩阵内存占用对比', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()

print(f"\n当序列长度为4096时，线性注意力节省内存: {(std_mems[-1] - linear_mems[-1]) / std_mems[-1] * 100:.2f}%")

## 7. 特征映射函数对比

In [ ]:
# 测试不同特征映射的效果
x_test = np.random.randn(100, 32)

# 标准注意力
std_attn = StandardAttention(32)
output_std = std_attn.forward(x_test)

# 不同特征映射
feature_maps = ['elu', 'rff']
results = {}

for fm in feature_maps:
    linear_attn = LinearAttention(32, feature_map=fm, num_features=64)
    
    start = time.time()
    output = linear_attn.forward(x_test)
    elapsed = time.time() - start
    
    error = np.mean(np.abs(output - output_std))
    results[fm] = {'time': elapsed, 'error': error}
    
    print(f"\n特征映射: {fm.upper()}")
    print(f"  计算时间: {elapsed*1000:.4f} ms")
    print(f"  平均误差: {error:.6f}")

## 8. 总结

### 线性注意力的优势

1. **复杂度降低**: O(n²) → O(n)
2. **内存节省**: 特别是在长序列场景下
3. **速度提升**: 序列越长，加速效果越明显
4. **可扩展性**: 能处理更长的序列

### 权衡

1. **精度损失**: 是标准注意力的近似
2. **特征映射选择**: 需要根据任务调整
3. **实现复杂度**: 需要理解kernel trick

### 应用场景

- 长文档处理
- 视频理解（长时序）
- 蛋白质序列分析
- 任何需要处理超长序列的场景